In [76]:
%%bash
if [[ ! -d "./data" ]]
then
    echo "Missing extra files. Downloading..."
    git clone https://github.com/cs124/pa2-logistic-regression.git
    cp -r ./pa2-logistic-regression/{data,deps,util.py} .
fi

In [77]:
# install datasets from HuggingFace
!pip install -q datasets

Imports

In [78]:
# python standard library
from collections import defaultdict
import operator
import random
from typing import List, Dict, Tuple, Union

# third party modules
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

# for downloading and loading HuggingFace datasets
from datasets import load_dataset

# custom functions and classes
from util import load_data, Classifier, Example, evaluate, remove_stop_words
from util import check_logistic_loss, check_gradient_descent

In [79]:
# download yelp dataset
yelpDataset = load_dataset("Yelp/yelp_review_full")
print(yelpDataset)
print(type(yelpDataset))

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})
<class 'datasets.dataset_dict.DatasetDict'>


In [80]:
# see an example
example = yelpDataset["train"][1100]
print(f"Review text:\n{example["text"]}")
print(f"\nLabel:{example["label"]}")

Review text:
I really like this Northside institution.  They serve the beer in jars and I had Spaten German beer- very good.  They put really buttery free popcorn on the table so be careful- the grease sticks to your fingers.  In the bar they have an old fashioned player piano.  I really like the German fare- big portions. The schitzel is good but not great but overall way better than Hofbrauhaus and not Disney- it's the real McCoy.

Label:3


In [81]:
# taking subsample to train faster
'''
300,000: like 5-10 mins
650,000: 15 mins for subsample cell, 3 mins to train
650,000: max
'''
NUM_TRAIN_SAMPLES = 10000
# max is 50000
NUM_TEST_SAMPLES = 2000
SEED = 7

train_split = yelpDataset["train"].shuffle(seed=SEED).select(range(NUM_TRAIN_SAMPLES))
test_split = yelpDataset["test"].shuffle(seed=SEED).select(range(NUM_TEST_SAMPLES))

train_texts = train_split["text"]
train_labels = np.array(train_split["label"])
test_texts = test_split["text"]
test_labels = np.array(test_split["label"])

# create a dev set from train
dev_fraction = 0.1
num_dev = int(len(train_texts) * dev_fraction)

shuffled_indices = np.random.permutation(len(train_texts))
dev_indices = shuffled_indices[:num_dev]
train_indices = shuffled_indices[num_dev:]

dev_texts = [train_texts[int(i)] for i in dev_indices]
dev_labels = [train_labels[i] for i in dev_indices]

train_texts = [train_texts[int(i)] for i in train_indices]
train_labels = train_labels[train_indices]

print(f"Train examples: {len(train_texts)}")
print(f"Dev examples: {len(dev_texts)}")
print(f"Test examples: {len(test_texts)}")
print(f"Label distribution (train): {np.bincount(train_labels)}")
print(f"Label distribution (dev): {np.bincount(dev_labels)}")

Train examples: 9000
Dev examples: 1000
Test examples: 2000
Label distribution (train): [1807 1752 1828 1819 1794]
Label distribution (dev): [209 205 189 205 192]


Turning each rating label into a one-hot vector

In [82]:
def one_hot(y: np.ndarray, num_classes: int) -> np.ndarray:
  """
  Converts an array of integers to a one-hot encoded array.

  Args:
    y: An array of integers.
    num_classes: The number of classes in the dataset.

  Returns:
    a 2D array of shape (num_examples, num_classes) where row i is all zeros except a 1 in column y[i].
  """
  m = y.shape[0]
  Y = np.zeros((m, num_classes))
  Y[np.arange(m), y] = 1
  return Y

In [83]:
# check
test_labels_small = np.array([0, 2, 4])
print(one_hot(test_labels_small, num_classes=5))

[[1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1.]]


Softmax function

In [84]:
def softmax(z: np.ndarray) -> np.ndarray:
  """
  Apply the softmax function row-wise.

  Args:
    z: a 2D numpy array of shape (num_examples, num_classes)

  Returns:
    a 2D array of the same shape, where each row is a probability distribution
  """
  z_shifted = z - np.max(z, axis=1, keepdims=True)
  exp_z = np.exp(z_shifted)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [85]:
# check: every row sums to 1
test_logits = np.array([[1, 2, 3], [0, 0, 0]])
probs = softmax(test_logits)
print(probs)
print("Row sums: ", probs.sum(axis=1))

[[0.09003057 0.24472847 0.66524096]
 [0.33333333 0.33333333 0.33333333]]
Row sums:  [1. 1.]


Corss-Entropy Loss

In [86]:
def cross_entropy_loss(y_pred: np.ndarray, y_true: np.ndarray) -> float:
  """
  Compute average multi-class cross-entropy loss

  Args:
    y_pred: predicted probabilities
    y_true: true labels

  Returns:
    average multi-class cross-entropy loss as a float
  """
  epsilon = 1e-8
  m = y_true.shape[0]
  total_loss = -np.sum(y_true * np.log(y_pred + epsilon))
  return total_loss / m

In [87]:
# check loss function
def print_loss(y_pred, y_true):
    print("Predicted = {}, True = {} : Loss = {}".format(
          y_pred, y_true, cross_entropy_loss(np.array([y_pred]),
                                        np.array([y_true]))))

print_loss(0.0, 1)
print_loss(0.5, 1)
print_loss(1, 1)

Predicted = 0.0, True = 1 : Loss = 18.420680743952367
Predicted = 0.5, True = 1 : Loss = 0.6931471605599454
Predicted = 1, True = 1 : Loss = -9.999999889225291e-09


Logistic Regression Gradient Descent

In [95]:
def gradient_descent(X: np.ndarray,
                     Y_onehot: np.ndarray,
                     num_classes: int,
                     X_dev: np.ndarray = None,
                     y_dev: np.array = None,
                     Y_dev_onehot: np.ndarray = None,
                     batch_size: int = 2000,
                     learning_rate: float = 0.5,
                     num_epochs: int = 1000,
                     print_every: int = 100,
                     epsilon: float = 1e-8) -> Tuple[np.ndarray, np.ndarray]:
    m, n = X.shape
    W = np.zeros((n, num_classes))
    b = np.zeros((num_classes,))
    loss = 0

    perm = np.random.permutation(m)
    X = X[perm]
    Y_onehot = Y_onehot[perm]

    batch_start = 0
    for i in range(num_iterations):
      batch_end = batch_start + batch_size
      if batch_end > m:
        perm = np.random.permutation(m)
        X = X[perm]
        Y_onehot = Y_onehot[perm]
        batch_start = 0
        batch_end = batch_size

      X_batch = X[batch_start:batch_end]
      Y_batch = Y_onehot[batch_start:batch_end]
      batch_start = batch_end

      batch_m = X_batch.shape[0]

      # forward pass
      Z = X_batch @ W + b
      P = softmax(Z)

      # calculate gradients
      dW = (1 / batch_m) * (X_batch.T @ (P - Y_batch))
      db = np.mean(P - Y_batch, axis=0)

      # update parameters
      W -= learning_rate * dW
      b -= learning_rate * db

      prev_loss = loss
      loss = cross_entropy_loss(P, Y_batch)
      if abs(prev_loss - loss) < epsilon:
          break

      if (i + 1) % print_every == 0:
          preds = np.argmax(P, axis=1)
          true = np.argmax(Y_batch, axis=1)
          acc = np.mean(preds == true)
          lossmsg = f"Epoch: {i + 1}/{num_iterations} | Batch Loss = {loss:.4f} | Batch Accuracy: {acc:.4f}"
          if X_dev is not None and y_dev is not None:
            dev_P = softmax(X_dev @ W + b)
            dev_loss = cross_entropy_loss(dev_P, Y_dev_onehot)
            dev_preds = np.argmax(dev_P, axis=1)
            dev_acc = np.mean(dev_preds == y_dev)
            lossmsg += f" | Dev Loss = {dev_loss:.4f} | Dev Accuracy: {dev_acc:.4f}"
          print(lossmsg)

    return W, b

The Classifier Class

In [96]:
class MulticlassLogisticRegressionClassifier:

    def __init__(self,
                 num_classes: int = 5,
                 min_df: int = 5,
                 batch_size: int = 2000,
                 learning_rate: float = 0.001,
                 num_iterations: int = 1000,
                 print_every: int = 100,
                 epsilon: float = 1e-8):
        self.num_classes = num_classes
        # min_df=5 makes it ignore words that appear in fewer than 5 documents
        self.vectorizer = CountVectorizer(min_df=min_df)
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.num_iterations = num_iterations
        self.print_every = print_every
        self.epsilon = epsilon
        self.W = None
        self.b = None

    def train(self, texts: List[str], labels: List[int],
              dev_texts: List[str] = None, dev_labels: List[int] = None) -> None:
        # turns text into counts of words
        X = self.vectorizer.fit_transform(texts).toarray()
        Y = np.array(labels)
        Y_onehot = one_hot(Y, self.num_classes)

        X_dev = Y_dev_onehot = y_dev = None
        if dev_texts is not None and dev_labels is not None:
            X_dev = self.vectorizer.transform(dev_texts).toarray()
            y_dev = np.array(dev_labels)
            Y_dev_onehot = one_hot(y_dev, self.num_classes)

        self.W, self.b = gradient_descent(
            X, Y_onehot, self.num_classes,
            X_dev = X_dev, y_dev = y_dev, Y_dev_onehot = Y_dev_onehot,
            batch_size=self.batch_size,
            learning_rate=self.learning_rate,
            num_iterations=self.num_iterations,
            print_every=self.print_every,
            epsilon=self.epsilon,
        )

    def predict(self, texts: List[str], return_scores: bool = False):
        """
        Predict star ratings for new sample text

        Args:
            texts: list of raw review strings
            return_scores: if True, return the full probability distribution
                over classes for each example instead of the single best
                class

        Returns:
            If return_scores is False: a 1D array of predicted class indices
            (0-4), shape (num_examples,)
            If return_scores is True: a 2D array of predicted probabilities,
            shape (num_examples, num_classes)
        """
        # converts new review into count of words (using same vocabulary, so not fit_ and only transform)
        X = self.vectorizer.transform(texts).toarray()
        P = softmax(X @ self.W + self.b)
        if return_scores:
            return P
        return np.argmax(P, axis=1)

    def get_weights(self) -> np.ndarray:
        """Return the learned weight matrix, shape (num_features, num_classes)"""
        return self.W

Training

In [99]:
clf = MulticlassLogisticRegressionClassifier(
    num_classes = 5,
    min_df = 5,
    batch_size = 2000,
    learning_rate = 0.05,
    num_iterations = 300,
    print_every = 30,
)

clf.train(train_texts, train_labels, dev_texts = dev_texts, dev_labels = dev_labels)

Epoch: 30/300 | Batch Loss = 1.5035 | Batch Accuracy: 0.3445 | Dev Loss = 1.5607 | Dev Accuracy: 0.4120
Epoch: 60/300 | Batch Loss = 1.4897 | Batch Accuracy: 0.4510 | Dev Loss = 1.4222 | Dev Accuracy: 0.4060
Epoch: 90/300 | Batch Loss = 1.3575 | Batch Accuracy: 0.4640 | Dev Loss = 1.3658 | Dev Accuracy: 0.4480
Epoch: 120/300 | Batch Loss = 1.3982 | Batch Accuracy: 0.4860 | Dev Loss = 1.3270 | Dev Accuracy: 0.4680
Epoch: 150/300 | Batch Loss = 1.3447 | Batch Accuracy: 0.4950 | Dev Loss = 1.3038 | Dev Accuracy: 0.4720
Epoch: 180/300 | Batch Loss = 1.3143 | Batch Accuracy: 0.4920 | Dev Loss = 1.2929 | Dev Accuracy: 0.4780
Epoch: 210/300 | Batch Loss = 1.2757 | Batch Accuracy: 0.5120 | Dev Loss = 1.3057 | Dev Accuracy: 0.4760
Epoch: 240/300 | Batch Loss = 1.2337 | Batch Accuracy: 0.4930 | Dev Loss = 1.3032 | Dev Accuracy: 0.4790
Epoch: 270/300 | Batch Loss = 1.2237 | Batch Accuracy: 0.5435 | Dev Loss = 1.2482 | Dev Accuracy: 0.5040
Epoch: 300/300 | Batch Loss = 1.1966 | Batch Accuracy: 0.5

Testing accuracy

In [91]:
def accuracy(preds, labels) -> float:
    return float(np.mean(np.array(preds) == np.array(labels)))

train_preds = clf.predict(train_texts, return_scores=False)
test_preds = clf.predict(test_texts, return_scores=False)

print("Train accuracy:", accuracy(train_preds, train_labels))
print("Test accuracy:", accuracy(test_preds, test_labels))

Train accuracy: 0.4131111111111111
Test accuracy: 0.3945


Using the model!!

In [92]:
def show_predictions(clf, reviews: List[str]) -> None:
    scores = clf.predict(reviews, return_scores=True)
    preds = np.argmax(scores, axis=1)
    for text, pred, score in zip(reviews, preds, scores):
        print(f"Predicted stars: {pred + 1}  (confidence: {score[pred]:.2f})")
        print(f"Review: {text[:120]}...")
        print("-" * 60)

my_reviews = [
    "Absolutely loved this place, the staff was so friendly and the food was amazing!",
    "It was fine, nothing special, but nothing terrible either.",
    "Worst experience of my life, the food was cold and the service was awful.",
]
show_predictions(clf, my_reviews)

Predicted stars: 2  (confidence: 0.21)
Review: Absolutely loved this place, the staff was so friendly and the food was amazing!...
------------------------------------------------------------
Predicted stars: 3  (confidence: 0.21)
Review: It was fine, nothing special, but nothing terrible either....
------------------------------------------------------------
Predicted stars: 2  (confidence: 0.21)
Review: Worst experience of my life, the food was cold and the service was awful....
------------------------------------------------------------


yay the end